In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(
    model='gemini-3.1-flash-lite',
    google_api_key=os.getenv('GEMINI_API_KEY')
)

In [3]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [4]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [5]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [6]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()


In [7]:
intial_state = {'title': 'about Quantiphi'}

final_state = workflow.invoke(intial_state)

print(final_state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'title': 'about Quantiphi', 'outline': [{'type': 'text', 'text': 'This outline is designed for a corporate, professional blog post that positions **Quantiphi** as a leader in the AI-first digital engineering space.\n\n---\n\n### Blog Title Ideas:\n*   *Quantiphi: Scaling Digital Transformation Through AI-First Engineering*\n*   *Beyond the Hype: How Quantiphi is Reshaping Enterprise Intelligence*\n*   *An Inside Look at Quantiphi: Solving Complex Problems with Data and AI*\n\n---\n\n### Blog Outline\n\n#### I. Introduction\n*   **The Hook:** Briefly mention the current state of the AI revolution—how businesses are struggling to move from experimentation to enterprise-scale deployment.\n*   **Defining Quantiphi:** Introduce Quantiphi as an AI-first digital engineering company that helps organizations solve their toughest business problems through data, analytics, and machine learning.\n*   **The Thesis Statement:** Why Quantiphi’s unique blend of "deep tech" and "domain expertise" make

In [8]:
print(final_state['outline'])

[{'type': 'text', 'text': 'This outline is designed for a corporate, professional blog post that positions **Quantiphi** as a leader in the AI-first digital engineering space.\n\n---\n\n### Blog Title Ideas:\n*   *Quantiphi: Scaling Digital Transformation Through AI-First Engineering*\n*   *Beyond the Hype: How Quantiphi is Reshaping Enterprise Intelligence*\n*   *An Inside Look at Quantiphi: Solving Complex Problems with Data and AI*\n\n---\n\n### Blog Outline\n\n#### I. Introduction\n*   **The Hook:** Briefly mention the current state of the AI revolution—how businesses are struggling to move from experimentation to enterprise-scale deployment.\n*   **Defining Quantiphi:** Introduce Quantiphi as an AI-first digital engineering company that helps organizations solve their toughest business problems through data, analytics, and machine learning.\n*   **The Thesis Statement:** Why Quantiphi’s unique blend of "deep tech" and "domain expertise" makes them a partner of choice for Fortune 5

In [9]:
print(final_state['content'])

[{'type': 'text', 'text': '# Beyond the Hype: How Quantiphi is Reshaping Enterprise Intelligence\n\nThe current state of the artificial intelligence revolution is paradoxical. While nearly every enterprise leader is eager to integrate AI into their business model, a significant gap remains between high-level experimentation and meaningful, enterprise-scale deployment. Many organizations find themselves stuck in "pilot purgatory," struggling to move beyond proof-of-concepts to realize actual ROI.\n\nThis is where **Quantiphi** enters the conversation. As an AI-first digital engineering company, Quantiphi has spent the last decade positioning itself not just as a service provider, but as a strategic partner capable of solving the toughest business problems through data, analytics, and machine learning.\n\n### The Origin Story: Bridging Data and Intelligence\nQuantiphi was founded on a singular vision: to bridge the chasm between raw data and actionable intelligence. The founders recogniz